# 02 · Retrieval pipeline, evaluation, and the routing experiment

Path A (Text2Cypher over the graph), Path B (dense retrieval over discharge
summaries), the 44-question evaluation, and the routing-versus-concatenation
experiment.

**Requires:** `01_build_graph.ipynb` to have run; an Anthropic API key; and, for
Path B only, PhysioNet credentialing for MIMIC-IV-Note with Google BigQuery
access granted.

**Run top to bottom.** Sections 1–3 need no credentialed data. Sections 4–6 do.

`evaluation.py`, `experiment.py` and `pipeline.py` live in the repository and are
loaded with `exec` rather than pasted, so the notebook and the repository cannot
drift apart. Upload them to the Colab file panel first.

**Cell outputs are stripped before committing.** Path B prints discharge-summary
text, which is credentialed and must not be redistributed.

## 1 · Connect

In [ ]:
!pip install -q neo4j anthropic

import anthropic
from neo4j import GraphDatabase
from getpass import getpass

URI  = input("AuraDB connection URI: ").strip()
USER = input("username (instance id): ").strip()
DB   = USER

driver = GraphDatabase.driver(URI, auth=(USER, getpass("Neo4j password: ")))
driver.verify_connectivity()

client = anthropic.Anthropic(api_key=getpass("Anthropic API key: ").strip())
MODEL  = "claude-opus-5"


def cypher(q, **params):
    with driver.session(database=DB) as s:
        return [r.data() for r in s.run(q, **params)]


print("connected ·", cypher("MATCH (n) RETURN count(n) AS n")[0]["n"], "nodes")

## 2 · Path A — router, Text2Cypher, validator

`temperature=0` is passed through `extra_body` because some SDK versions do not
expose it as a named argument. Without it, evaluation runs are not reproducible.

`llm()` raises on `stop_reason == "max_tokens"`. A silently truncated completion
becomes invalid Cypher, which then looks like a model failure rather than a
budget one — that mistake cost two questions in an earlier run.

In [ ]:
exec(open('pipeline.py').read())

print(route("How many admissions did patient 10014354 have?"))   # STRUCTURED
print(route("Why was venetoclax started instead of prednisone?")) # NARRATIVE

## 3 · Evaluate Path A

`verify_gold()` executes every reference query and costs no API calls. Run it first: a reference query that errors is a bug in the bank, not a model failure.

In [ ]:
exec(open('evaluation.py').read())

verify_gold()

In [ ]:
results = evaluate()
report(results)
save(results)
failures(results)

Five runs for a range. One unstable question moves a tier by 10 points, so a single run is not a measurement.

In [ ]:
import statistics as st, json

RUNS = 5
runs = []
for k in range(RUNS):
    rr = evaluate(verbose=False)
    if any(str(r.get("got_route", "")).startswith("ERROR") for r in rr):
        print(f"run {k+1}: API errors — discarding")
        break
    runs.append(rr)
    json.dump(rr, open(f"run_{k}.json", "w"), indent=2, default=str)
    print(f"run {k+1}/{RUNS} complete")

for key, crit in (("verdict_strict", "strict"), ("verdict", "projection")):
    print(f"\n{crit}")
    for tier in ("simple", "medium", "hard"):
        ids = [q["id"] for q in QUESTIONS if q["tier"] == tier and q.get("gold")]
        accs = [100 * sum(r[key] in OK for r in rr if r["id"] in ids) / len(ids)
                for rr in runs]
        print(f"  {tier:<8}{st.mean(accs):>7.1f}%  ({min(accs):.0f}-{max(accs):.0f})")

## 4 · Path B — discharge summaries

**Credentialed from here on.** MIMIC-IV-Note requires PhysioNet credentialing
and a BigQuery access grant on the Google account used below. Nothing retrieved
here may be committed or placed in a shared Drive folder.

In [ ]:
!gcloud auth application-default login --no-launch-browser

In [ ]:
PROJECT = input("GCP project id: ").strip()
!gcloud auth application-default set-quota-project {PROJECT}

from google.cloud import bigquery
bq = bigquery.Client(project=PROJECT)
print([t.table_id for t in bq.list_tables("physionet-data.mimiciv_note")])

Extract only the demo cohort — a few megabytes rather than the 1.1 GB file.

In [ ]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

import os
NOTE_DIR = '/content/drive/MyDrive/graphrag_clinical/note'
os.makedirs(NOTE_DIR, exist_ok=True)

sids = sorted(r["p"] for r in cypher("MATCH (p:Patient) RETURN p.subject_id AS p"))

notes = bq.query(f"""
SELECT subject_id, hadm_id, note_id, note_type, note_seq, charttime, storetime, text
FROM `physionet-data.mimiciv_note.discharge`
WHERE subject_id IN ({','.join(map(str, sids))})
ORDER BY subject_id, charttime
""").to_dataframe()

adm = cypher("MATCH (a:Admission) RETURN count(a) AS n")[0]["n"]
cov = notes.hadm_id.nunique()
print(f"{len(notes)} discharge summaries")
print(f"{notes.subject_id.nunique()} of {len(sids)} patients")
print(f"{cov} of {adm} admissions covered ({100*cov/adm:.0f}%)")

notes.to_parquet(f"{NOTE_DIR}/demo_notes.parquet")

Chunk and embed. Embeddings are computed **locally**, so note text never leaves this VM. Set the runtime to a GPU (Runtime → Change runtime type → T4) or this takes about 35 minutes instead of one.

In [ ]:
!pip install -q sentence-transformers

import numpy as np, torch
from sentence_transformers import SentenceTransformer

CHUNK, OVERLAP = 1200, 200

rows = []
for r in notes.itertuples():
    for i in range(0, len(r.text), CHUNK - OVERLAP):
        piece = r.text[i:i + CHUNK].strip()
        if len(piece) > 100:
            rows.append({"subject_id": int(r.subject_id), "hadm_id": int(r.hadm_id),
                         "note_id": r.note_id, "offset": i, "text": piece})
chunks = pd.DataFrame(rows)
print(f"{len(chunks)} chunks from {len(notes)} notes")

model = SentenceTransformer("BAAI/bge-small-en-v1.5",
                            device="cuda" if torch.cuda.is_available() else "cpu")
emb = model.encode(chunks.text.tolist(), batch_size=64,
                   normalize_embeddings=True, show_progress_bar=True)

np.save(f"{NOTE_DIR}/emb.npy", emb)
chunks.to_parquet(f"{NOTE_DIR}/chunks.parquet")


def search(question, k=5, hadm_id=None, subject_id=None):
    """Exact cosine search. Scope with hadm_id or subject_id — that scoping is
    how Path A narrows the space before Path B runs."""
    mask = np.ones(len(chunks), bool)
    if hadm_id    is not None: mask &= (chunks.hadm_id    == hadm_id).values
    if subject_id is not None: mask &= (chunks.subject_id == subject_id).values
    if not mask.any():
        return chunks.iloc[:0].assign(score=[])
    q = model.encode([question], normalize_embeddings=True)[0]
    sims = emb[mask] @ q
    idx  = np.argsort(-sims)[:k]
    out  = chunks[mask].iloc[idx].copy()
    out["score"] = sims[idx]
    return out[["score", "subject_id", "hadm_id", "note_id", "text"]]


print(emb.shape)

## 5 · Routing versus context concatenation

Both arms use the same synthesis prompt and the same model, so the comparison
isolates context composition rather than retrieval quality.

In [ ]:
exec(open('experiment.py').read())

pairs = eligible()          # no API cost
print(f"{len(pairs)} questions with a reference answer small enough to score in prose")

results_x = run_experiment(pairs)
report(results_x)
save(results_x)

The sensitive version: value recall over every question with a reference
answer, plus the within-arm control. If re-running one arm differs from itself
by as much as the two arms differ, there is no detectable effect.

In [ ]:
import statistics as st

all_pairs = []
for q in QUESTIONS:
    if q.get("gold") and not q.get("expect_empty"):
        try:
            all_pairs.append((q, cypher(q["gold"])))
        except Exception:
            pass

A = run_experiment(all_pairs, verbose=False)
B = run_experiment(all_pairs, verbose=False)

for rr in (A, B):
    for x in rr:
        for arm in ("R", "C"):
            x[arm]["recall"] = value_recall(x["gold_rows"], x[arm]["answer"])

bi = {x["id"]: x for x in B}
within = [abs(x["R"]["recall"] - bi[x["id"]]["R"]["recall"]) for x in A]
between = ([abs(x["C"]["recall"] - x["R"]["recall"]) for x in A] +
           [abs(x["C"]["recall"] - x["R"]["recall"]) for x in B])

print(f"mean value recall   routing {st.mean(x['R']['recall'] for x in A):.3f}"
      f"   concat {st.mean(x['C']['recall'] for x in A):.3f}")
print(f"\nmean |delta| per question")
print(f"  routing vs itself   {st.mean(within):.3f}")
print(f"  routing vs concat   {st.mean(between):.3f}")

## 6 · One question end to end

`ask()` prints every stage — route, generated Cypher, validation, row count,
answer — so the retrieval step is inspectable rather than opaque. This is the
auditability argument in concrete form.

In [ ]:
ask("Which medications did patient 10014354 receive during the admission "
    "where creatinine was highest?", sid=10014354)